In [5]:
# %%
# =============================================================================
# Paper C · 01_build_graph  (MEASURED schema; dual masks A + B)
# Non-financial (lawsuit + relation) bankruptcy prediction
#
# Build/validate a PyG HeteroData graph for 02–06. No training. CPU is enough.
#
# ── MEASURED schema (verified from the actual pickles) ───────────────────────
# Global node space: 6,381 ids (0..6380, 11 gaps). companies=3,976 persons=2,405.
# node2index : {int global_id: "中文名"}  — names are identifiers only, NOT features.
#
# Each {train,validate,test}_data.pkl is a length-5 list for that split's
# labelled companies (2816 / 721 / 491 rows):
#   [0] risk_data    : {global_company_id: [[cause,court,category,time_m,time_label], ...]}
#   [1] company_attr : [[register_capital, paid_capital, setup_time_m], ...] (split rows)
#   [2] hete_graph   : (edge_index[2xN], edge_type[N], edge_weight[N]) in GLOBAL ids
#   [3] hyp_graph    : {industry:{...}, area:{...}, qualify:{...}}
#   [4] label        : 0/1 list, length = this split's company count
# split_data_idx.pkl : [train_ids, valid_ids, test_ids] as GLOBAL company ids,
#                      row-aligned to each split's company_attr / label.
# meta_emb.pkl       : [company_emb (3976x32), person_emb (2405x32)] metapath2vec.
#
# ── KEY FACT that shaped this file ───────────────────────────────────────────
# 52 companies appear in TWO splits under the SAME global id (verified: id 2058
# is the same firm in train and valid). Time-based splits are therefore NOT
# disjoint out of the box. We build TWO label-mask sets on ONE shared graph:
#   • MASK A (strict, no-leakage, DEFAULT): keep each duplicated company only in
#     its LATEST split (priority test > valid > train); remove from earlier ones.
#   • MASK B (reproduction): keep split_data_idx exactly as shipped, so the
#     original code / container can reproduce the paper's numbers.
# Graph nodes/edges are identical across A and B; only the label masks differ.
# =============================================================================

import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import torch

# %%
# --- Paths -------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
DATA_DIR = ROOT / "data" / "external" / "ComRisk" / "data"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.exists(), f"SMEsD data dir missing: {DATA_DIR} — run 00 first"


def load(name):
    with open(DATA_DIR / name, "rb") as f:
        return pickle.load(f)


# %%
# --- 1. Auxiliary files ------------------------------------------------------
node2index = load("node2index.pkl")     # {int global_id: "name"}
split_idx = load("split_data_idx.pkl")   # [train_ids, valid_ids, test_ids] global
meta_emb = load("meta_emb.pkl")          # [company_emb, person_emb]

_k = next(iter(node2index))
name2idx = ({str(k): int(v) for k, v in node2index.items()} if isinstance(_k, str)
            else {str(v): int(k) for k, v in node2index.items()})
idx2name = {v: k for k, v in name2idx.items()}      # display only (07_case_study)

company_emb = np.asarray(meta_emb[0], dtype=np.float32)
person_emb = np.asarray(meta_emb[1], dtype=np.float32)
N_COMPANY, N_PERSON = company_emb.shape[0], person_emb.shape[0]

all_ids = np.fromiter(name2idx.values(), dtype=np.int64)
GLOBAL_N = int(all_ids.max()) + 1
print(f"nodes: companies={N_COMPANY:,} persons={N_PERSON:,} "
      f"global_slots={GLOBAL_N:,} (gaps={GLOBAL_N - len(all_ids)})")


# %%
# --- 2. Load splits, confirm length-5 + row alignment ------------------------
FIELDS = ["risk_data", "company_attr", "hete_graph", "hyp_graph", "label"]
splits = {}
for nm, fn in [("train", "train_data.pkl"),
               ("valid", "validate_data.pkl"),
               ("test", "test_data.pkl")]:
    obj = load(fn)
    assert isinstance(obj, (list, tuple)) and len(obj) == 5, f"{fn}: expected length-5"
    splits[nm] = dict(zip(FIELDS, obj))
    ca = np.asarray(splits[nm]["company_attr"])
    lab = np.asarray(splits[nm]["label"])
    assert ca.shape[0] == lab.shape[0], f"{nm}: attr rows != label length"
    assert ca.shape[1] == 3, f"{nm}: company_attr must be 3 cols (no financial ratios)"
    print(f"{nm:5s}: companies={ca.shape[0]:,} pos={int(lab.sum()):,} ({lab.mean():.2%})")

split_global_ids = {nm: np.asarray(ids, dtype=np.int64)
                    for nm, ids in zip(["train", "valid", "test"], split_idx)}
for nm in ["train", "valid", "test"]:
    assert np.asarray(splits[nm]["company_attr"]).shape[0] == split_global_ids[nm].shape[0], \
        f"{nm}: company_attr rows != split_idx length"


# %%
# --- 3. Quantify the cross-split duplication (do NOT silently drop) ----------
# We treat duplicates explicitly and record which ids collide, for the report
# and for the strict-mask construction below.
tr = split_global_ids["train"]; va = split_global_ids["valid"]; te = split_global_ids["test"]
sets = {"train": set(tr.tolist()), "valid": set(va.tolist()), "test": set(te.tolist())}
dup_tv = sets["train"] & sets["valid"]
dup_tt = sets["train"] & sets["test"]
dup_vt = sets["valid"] & sets["test"]
all_dups = dup_tv | dup_tt | dup_vt
print(f"cross-split duplicated companies: {len(all_dups)} "
      f"(train∩valid={len(dup_tv)} train∩test={len(dup_tt)} valid∩test={len(dup_vt)})")
print(f"union of labelled companies: {len(sets['train'] | sets['valid'] | sets['test']):,}")

# Sanity: confirm duplicates are the SAME firm (same name) — true collision.
for g in list(all_dups)[:3]:
    print(f"  id {g}: {idx2name.get(g, '??')}")


# %%
# --- 4. Lawsuit encoding cardinalities (shared width across splits) ----------
cause, court, category = set(), set(), set()
for s in splits.values():
    for events in s["risk_data"].values():
        for ev in events:                # [cause, court, category, time_m, time_label]
            cause.add(int(ev[0])); court.add(int(ev[1])); category.add(int(ev[2]))
N_CAUSE, N_COURT, N_CAT = max(cause) + 1, max(court) + 1, max(category) + 1
RISK_W = N_CAUSE + N_COURT + N_CAT + 1
print(f"lawsuit codes: cause={N_CAUSE} court={N_COURT} category={N_CAT} "
      f"→ risk width={RISK_W}")


# %%
# --- 5. GLOBAL company feature matrix ----------------------------------------
# [3 basic | cause_counts | court_counts | category_counts | mean_time].
# For the 52 duplicates, later splits overwrite earlier ones when scattering;
# since company_attr for a given global id is the same firm, this is consistent.
# (If a duplicate's attrs ever differ across splits we surface it below.)
BASIC_W = 3
company_x = np.zeros((GLOBAL_N, BASIC_W + RISK_W), dtype=np.float32)
company_mask = np.zeros(GLOBAL_N, dtype=bool)

# consistency check for duplicated firms' basic attrs
attr_by_id = {}
for nm in ["train", "valid", "test"]:
    ca = np.asarray(splits[nm]["company_attr"], dtype=np.float32)
    for row, gid in enumerate(split_global_ids[nm]):
        gid = int(gid)
        if gid in attr_by_id and not np.allclose(attr_by_id[gid], ca[row]):
            print(f"  ⚠️ dup id {gid} has differing attrs across splits "
                  f"{attr_by_id[gid]} vs {ca[row]}")
        attr_by_id[gid] = ca[row]

for nm in ["train", "valid", "test"]:
    ca = np.asarray(splits[nm]["company_attr"], dtype=np.float32)
    gids = split_global_ids[nm]
    company_x[gids, :BASIC_W] = ca
    company_mask[gids] = True
    for gid, events in splits[nm]["risk_data"].items():
        gid = int(gid)
        cause_c = np.zeros(N_CAUSE, np.float32)
        court_c = np.zeros(N_COURT, np.float32)
        cat_c = np.zeros(N_CAT, np.float32)
        times = []
        for ev in events:
            cause_c[int(ev[0])] += 1
            court_c[int(ev[1])] += 1
            cat_c[int(ev[2])] += 1
            times.append(float(ev[3]))
        company_x[gid, BASIC_W:] = np.concatenate(
            [cause_c, court_c, cat_c, [np.mean(times) if times else 0.0]])

print(f"company_x: {company_x.shape}  labelled slots: {company_mask.sum():,}")


# %%
# --- 6. Labels + DUAL split masks (A strict, B reproduction) -----------------
# Global label vector: a firm's label is identical across splits (same firm), so
# scatter is unambiguous. Only the MASKS differ between A and B.
y = np.full(GLOBAL_N, -1, dtype=np.int64)
for nm in ["train", "valid", "test"]:
    y[split_global_ids[nm]] = np.asarray(splits[nm]["label"], dtype=np.int64)

def empty_masks():
    return {k: np.zeros(GLOBAL_N, dtype=bool) for k in ["train", "valid", "test"]}

# --- MASK B: reproduction — split_data_idx exactly as shipped (may overlap) ---
mask_B = empty_masks()
for nm in ["train", "valid", "test"]:
    mask_B[nm][split_global_ids[nm]] = True

# --- MASK A: strict — each duplicated id kept only in its LATEST split ---------
# priority test > valid > train: assign, then remove from earlier splits.
mask_A = empty_masks()
assigned = np.zeros(GLOBAL_N, dtype=bool)
for nm in ["test", "valid", "train"]:            # latest first
    gids = split_global_ids[nm]
    take = gids[~assigned[gids]]                 # only ids not yet claimed
    mask_A[nm][take] = True
    assigned[take] = True

# verify A is disjoint; B may not be
for tag, M in [("A", mask_A), ("B", mask_B)]:
    ov = int((M["train"] & M["valid"]).sum()
             + (M["train"] & M["test"]).sum()
             + (M["valid"] & M["test"]).sum())
    n = {k: int(M[k].sum()) for k in M}
    print(f"mask {tag}: train={n['train']:,} valid={n['valid']:,} test={n['test']:,} "
          f"overlap={ov}" + ("  (disjoint ✓)" if ov == 0 else "  (overlap kept)"))

ov_A = int((mask_A["train"] & mask_A["valid"]).sum()
           + (mask_A["train"] & mask_A["test"]).sum()
           + (mask_A["valid"] & mask_A["test"]).sum())
assert ov_A == 0, "mask A must be disjoint"


# %%
# --- 7. Parse + merge heterogeneous edges (GLOBAL ids) -----------------------
def parse_hete(hg):
    ei, et, ew = (hg[0], hg[1], hg[2]) if isinstance(hg, (list, tuple)) \
        else (hg["edge_index"], hg["edge_type"], hg["edge_weight"])
    ei = np.asarray(ei)
    if ei.shape[0] != 2:
        ei = ei.T
    return ei.astype(np.int64), np.asarray(et).astype(np.int64), np.asarray(ew, np.float32)

edges = {}
for nm in ["train", "valid", "test"]:
    ei, et, ew = parse_hete(splits[nm]["hete_graph"])
    for k in range(ei.shape[1]):
        edges[(int(ei[0, k]), int(ei[1, k]))] = (int(et[k]), float(ew[k]))

src = np.array([s for s, _ in edges], np.int64)
dst = np.array([d for _, d in edges], np.int64)
etype = np.array([v[0] for v in edges.values()], np.int64)
eweight = np.array([v[1] for v in edges.values()], np.float32)
rel_ids = sorted(set(etype.tolist()))
print(f"merged edges: {len(edges):,}  relations={rel_ids}")
print("per relation:", dict(sorted(Counter(etype.tolist()).items())))


# %%
# --- 8. Assemble PyG HeteroData (shared graph; both masks attached) ----------
# person mapping FIX: persons occupy global ids [N_COMPANY, GLOBAL_MAX] by
# convention (verified). Map by global-id OFFSET so rows align 1:1 with
# person_emb, INCLUDING 3 isolated persons (emb present, no edges: 3979/4144/4683).
from torch_geometric.data import HeteroData

comp_global = np.where(company_mask)[0]
comp_g2l = {int(g): i for i, g in enumerate(comp_global)}

# Person universe = ALL emb rows, indexed by offset (not by edge appearance).
person_global = np.arange(N_COMPANY, N_COMPANY + N_PERSON, dtype=np.int64)
pers_g2l = {int(g): i for i, g in enumerate(person_global)}   # local = global - N_COMPANY
print(f"typed nodes: company={len(comp_global):,} person={len(person_global):,} "
      f"(offset-aligned to person_emb; 3 isolated persons retained)")

def sel(mask_dict, split, gids):
    return torch.tensor(mask_dict[split][gids])

data = HeteroData()
data["company"].x = torch.tensor(company_x[comp_global], dtype=torch.float32)
data["company"].y = torch.tensor(y[comp_global], dtype=torch.long)
data["company"].train_mask = sel(mask_A, "train", comp_global)
data["company"].val_mask = sel(mask_A, "valid", comp_global)
data["company"].test_mask = sel(mask_A, "test", comp_global)
data["company"].train_mask_repro = sel(mask_B, "train", comp_global)
data["company"].val_mask_repro = sel(mask_B, "valid", comp_global)
data["company"].test_mask_repro = sel(mask_B, "test", comp_global)
data["company"].num_nodes = len(comp_global)

# Now person_emb aligns exactly (2405 rows <-> 2405 offset-indexed persons).
assert person_emb.shape[0] == len(person_global), "person_emb must align by offset"
data["person"].x = torch.tensor(person_emb, dtype=torch.float32)
data["person"].num_nodes = len(person_global)

def type_local(gid):
    return ("company", comp_g2l[gid]) if gid in comp_g2l else ("person", pers_g2l[gid])

buckets = {}
for s, d, r, w in zip(src, dst, etype, eweight):
    st, sl = type_local(int(s)); dt, dl = type_local(int(d))
    buckets.setdefault((st, f"rel_{r}", dt), [[], [], []])
    buckets[(st, f"rel_{r}", dt)][0].append(sl)
    buckets[(st, f"rel_{r}", dt)][1].append(dl)
    buckets[(st, f"rel_{r}", dt)][2].append(w)

for (st, rel, dt), (s_l, d_l, w_l) in buckets.items():
    store = data[st, rel, dt]
    store.edge_index = torch.tensor([s_l, d_l], dtype=torch.long)
    store.edge_weight = torch.tensor(w_l, dtype=torch.float32)

print(data)


# %%
# --- 9. Persist for 02–06 ----------------------------------------------------
out = PROCESSED / "smesd_hetero.pt"
torch.save(data, out)
print(f"→ saved {out.relative_to(ROOT)} ({out.stat().st_size / 1e6:.1f} MB)")

torch.save({"comp_global": comp_global,
            "person_global": np.array(person_global),
            "idx2name": idx2name,
            "rel_ids": rel_ids,
            "dup_ids": np.array(sorted(all_dups), dtype=np.int64)},
           PROCESSED / "smesd_idmaps.pt")
print(f"→ saved {(PROCESSED / 'smesd_idmaps.pt').relative_to(ROOT)}")


# %%
# --- 10. Contract summary for downstream notebooks ---------------------------
print(f"""
Graph contract for 02–06
========================
  company.x[{data['company'].x.shape[1]}] = 3 basic + {RISK_W} lawsuit-aggregate ; y 0/1
  masks (DEFAULT = A, strict no-leakage): train_mask / val_mask / test_mask
  masks (reproduction = B, as shipped)  : *_mask_repro
    → use A for the proposed Paper-C experiments (strict time split)
    → use B (or original container code) only to reproduce paper numbers
  person.x = metapath2vec 32-d ; edges: relations {rel_ids}, typed, edge_weight kept
  {len(all_dups)} duplicated companies recorded in smesd_idmaps.pt['dup_ids']

  RQ mapping
    RQ1 no financials → basic + graph only
    RQ2 intra (lawsuit-aggregate cols) vs contagion (message passing on edges)
    RQ3 pickles hold CODED cause ints only — NO raw lawsuit text.
        Confirm a raw-text source before 04_llm_lawsuit_struct.
""")
print("01_build_graph complete.")

nodes: companies=3,976 persons=2,405 global_slots=6,381 (gaps=11)
train: companies=2,816 pos=1,621 (57.56%)
valid: companies=721 pos=354 (49.10%)
test : companies=491 pos=318 (64.77%)
cross-split duplicated companies: 51 (train∩valid=35 train∩test=17 valid∩test=1)
union of labelled companies: 3,976
  id 900: 宁波冠霖国际贸易有限公司
  id 2058: 嵊州市舒凯电器科技有限公司
  id 2062: 台州金跃贸易有限公司
lawsuit codes: cause=11 court=4 category=4 → risk width=20
company_x: (6381, 23)  labelled slots: 3,976
mask A: train=2,765 valid=720 test=491 overlap=0  (disjoint ✓)
mask B: train=2,816 valid=721 test=491 overlap=53  (overlap kept)
merged edges: 13,336  relations=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
per relation: {0: 698, 1: 698, 2: 883, 3: 883, 4: 37, 5: 37, 6: 938, 7: 942, 8: 3657, 9: 3657, 10: 376, 11: 376, 12: 53, 13: 9, 14: 2, 15: 13, 16: 53, 17: 9, 18: 2, 19: 13}
typed nodes: company=3,976 person=2,405 (offset-aligned to person_emb; 3 isolated persons retained)
HeteroData(
  company